## 3 — GLMER-MRP (county-level estimates)
Frequentist 4-level hierarchical logistic regression (penalised ML) — Howe (2015) county architecture.

**Individual level:** `logit(p_i) = γ₀ + α_gender[g] + α_race[r] + α_educ[e] + α_county[c]`

**County level (informative prior driven by county-level covariates):**
`α_county[c] ~ N(α_state[s[c]] + γ_carbon·co2 + γ_pres·pres + γ_drive·drive + γ_samesex·samesex, σ_county²)`

**State level:** `α_state[s] ~ N(α_region[div[s]], σ_state²)` — bare nesting, no covariates

**Region level:** `α_region[r] ~ N(0, σ_region²)` — 9 Census divisions

Estimation: L-BFGS-B MAP with **diffuse priors** (σ_prior = 10, equivalent to penalised ML).

Output: `outputs/estimates/glmer_mrp_county_estimates.csv`

In [11]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.special import expit
from scipy.optimize import minimize

DATA_DIR   = Path("../test_data/processed/")
OUTPUT_DIR = Path("../outputs/")
OUTCOME    = "happening_bin"
MODEL_NAME = "glmer_mrp"
STATE_CSV_NAME = "glmer_mrp_state_estimates.csv"

STATE_NAMES = {
    "01":"Alabama","02":"Alaska","04":"Arizona","05":"Arkansas","06":"California",
    "08":"Colorado","09":"Connecticut","10":"Delaware","11":"District of Columbia",
    "12":"Florida","13":"Georgia","15":"Hawaii","16":"Idaho","17":"Illinois",
    "18":"Indiana","19":"Iowa","20":"Kansas","21":"Kentucky","22":"Louisiana",
    "23":"Maine","24":"Maryland","25":"Massachusetts","26":"Michigan",
    "27":"Minnesota","28":"Mississippi","29":"Missouri","30":"Montana",
    "31":"Nebraska","32":"Nevada","33":"New Hampshire","34":"New Jersey",
    "35":"New Mexico","36":"New York","37":"North Carolina","38":"North Dakota",
    "39":"Ohio","40":"Oklahoma","41":"Oregon","42":"Pennsylvania",
    "44":"Rhode Island","45":"South Carolina","46":"South Dakota",
    "47":"Tennessee","48":"Texas","49":"Utah","50":"Vermont",
    "51":"Virginia","53":"Washington","54":"West Virginia","55":"Wisconsin",
    "56":"Wyoming",
}

In [12]:
# ── Build county-level covariate table ─────────────────────────────────────
# poststrat_county already carries co2_per_capita and dem_share_two_party at the
# county level; merge in pct_drive_alone, pct_samesex_hh from the ACS extras.
ps_county = pd.read_csv(DATA_DIR / "poststrat_county.csv",
                         dtype={"county_fips": str, "state_fips": str})
acs_county = pd.read_csv(DATA_DIR / "acs_county_extra_covariates.csv",
                          dtype={"county_fips": str, "state_fips": str})

ps_county = ps_county.merge(
    acs_county[["county_fips", "pct_drive_alone", "pct_samesex_hh"]],
    on="county_fips", how="left",
)

county_cov = (
    ps_county[["county_fips", "state_fips",
               "co2_per_capita", "dem_share_two_party",
               "pct_drive_alone", "pct_samesex_hh"]]
    .drop_duplicates(subset=["county_fips"])
    .reset_index(drop=True)
)
# Impute any missing covariate values with the national mean
for c in ["co2_per_capita", "dem_share_two_party", "pct_drive_alone", "pct_samesex_hh"]:
    county_cov[c] = county_cov[c].fillna(county_cov[c].mean())

print(f"poststrat_county rows: {len(ps_county):,}  | counties: {ps_county['county_fips'].nunique():,}")
print(f"county covariate table: {county_cov.shape}")
print(county_cov.head(4).to_string(index=False))

poststrat_county rows: 99,940  | counties: 3,143
county covariate table: (3143, 6)
county_fips state_fips  co2_per_capita  dem_share_two_party  pct_drive_alone  pct_samesex_hh
      01001         01       81.134604             0.266411         0.843044        0.005769
      01003         01       10.888320             0.206524         0.794437        0.005769
      01005         01       12.489276             0.425850         0.832330        0.005769
      01007         01       10.493996             0.176151         0.848439        0.005769


In [13]:
# ── Load survey ───────────────────────────────────────────────────────────
survey = pd.read_csv(DATA_DIR / "climate_survey_responses_recoded.csv",
                     dtype={"state_fips": str, "county_fips": str})
survey = survey.dropna(subset=[OUTCOME]).copy()
survey[OUTCOME] = survey[OUTCOME].astype(float)
survey["educ_category"] = survey["educ_category"].astype(str)

print(f"Survey: {len(survey):,}  ({survey[OUTCOME].mean()*100:.1f}% Yes)")
print(f"Survey distinct counties: {survey['county_fips'].nunique():,}")

Survey: 1,011  (57.9% Yes)
Survey distinct counties: 403


In [14]:
# ── Build hierarchical indices ────────────────────────────────────────────
# county_cats: ALL counties from poststrat_county (~3,143)
# state_cats : ALL states  from poststrat_county (~51)
# div_cats   : 9 Census divisions (sourced from poststrat_county[DIVISION])
county_cats = sorted(ps_county["county_fips"].unique())
state_cats  = sorted(ps_county["state_fips"].unique())

# Each county_fips → its parent state_fips (ordered by county_cats)
cc = (county_cov.set_index("county_fips").loc[county_cats].reset_index())

county_state_idx = pd.Categorical(
    cc["state_fips"].astype(str),
    categories=state_cats,
).codes

# Each state_fips → its Census division (from poststrat_county.DIVISION)
state_div = (ps_county.groupby("state_fips")["DIVISION"].first()
                       .reset_index().rename(columns={"DIVISION": "division"}))
div_cats = sorted(state_div["division"].unique())
n_div = len(div_cats)

state_div_idx = pd.Categorical(
    state_div.set_index("state_fips").loc[state_cats]["division"].astype(str).values,
    categories=[str(d) for d in div_cats],
).codes

# Standardised county-level covariates (z-score, ordered to match county_cats)
def _std(x):
    return (x - x.mean()) / x.std()

co2_std     = _std(cc["co2_per_capita"].values)
pres_std    = _std(cc["dem_share_two_party"].values)
drive_std   = _std(cc["pct_drive_alone"].values)
samesex_std = _std(cc["pct_samesex_hh"].values)

n_county = len(county_cats)
n_s      = len(state_cats)

print(f"Counties: {n_county:,}  |  States: {n_s}  |  Divisions: {n_div}")
print(f"co2_std  range: [{co2_std.min():.2f}, {co2_std.max():.2f}]")
print(f"drive_std range: [{drive_std.min():.2f}, {drive_std.max():.2f}]")

Counties: 3,143  |  States: 51  |  Divisions: 9
co2_std  range: [-0.08, 53.72]
drive_std range: [-8.98, 2.53]


In [15]:
# ── Integer-encode survey categorical variables ───────────────────────────
gender_cats = sorted(survey["gender"].unique())
race_cats   = sorted(survey["race4"].unique())
educ_cats   = sorted(survey["educ_category"].unique())

g_idx = pd.Categorical(survey["gender"],        categories=gender_cats).codes
r_idx = pd.Categorical(survey["race4"],         categories=race_cats).codes
e_idx = pd.Categorical(survey["educ_category"], categories=educ_cats).codes
c_idx = pd.Categorical(survey["county_fips"],   categories=county_cats).codes
y     = survey[OUTCOME].values.astype(float)

# Drop any rows whose county_fips wasn't in poststrat_county (c_idx == -1)
mask = c_idx >= 0
if (~mask).sum():
    print(f"Dropping {(~mask).sum()} survey rows with county_fips missing from poststrat_county")
    g_idx, r_idx, e_idx, c_idx = g_idx[mask], r_idx[mask], e_idx[mask], c_idx[mask]
    y = y[mask]

n_g, n_r, n_e = len(gender_cats), len(race_cats), len(educ_cats)
n_obs = len(y)
n_p = 1 + 6 + 4 + n_g + n_r + n_e + n_div + n_s + n_county
print(f"gender:{n_g}  race:{n_r}  educ:{n_e}  div:{n_div}  state:{n_s}  county:{n_county}")
print(f"Total params: 1+6+4+{n_g}+{n_r}+{n_e}+{n_div}+{n_s}+{n_county} = {n_p}")
print(f"Observations: {n_obs:,}")

Dropping 2 survey rows with county_fips missing from poststrat_county
gender:2  race:4  educ:4  div:9  state:51  county:3143
Total params: 1+6+4+2+4+4+9+51+3143 = 3224
Observations: 1,009


### Fit — Howe (2015) 4-level MAP (diffuse priors = penalised ML)
Wide priors (σ_prior = 10) so regularisation is minimal — penalised maximum
likelihood rather than a full Bayesian posterior mode. Compare `4_GLMERStan_county`
which uses tight Bayesian priors.

In [16]:
# ─── Diffuse-prior MAP fit (Howe 2015 county architecture, 4-level) ──────────
SIGMA_PRIOR = 10.0   # wide = minimal shrinkage; GLMERStan uses 2.5

def _unpack(params):
    g0    = params[0]
    sigma = np.exp(params[1:7])  # [σ_gender, σ_race, σ_educ, σ_county, σ_state, σ_region]
    gc, gp, gd, gs = params[7], params[8], params[9], params[10]
    off = 11
    u_g   = params[off:off+n_g];        off += n_g
    u_r   = params[off:off+n_r];        off += n_r
    u_e   = params[off:off+n_e];        off += n_e
    u_reg = params[off:off+n_div];      off += n_div
    u_s   = params[off:off+n_s];        off += n_s
    u_c   = params[off:off+n_county]
    return g0, sigma, gc, gp, gd, gs, u_g, u_r, u_e, u_reg, u_s, u_c

def neg_log_posterior(params):
    g0, sigma, gc, gp, gd, gs, u_g, u_r, u_e, u_reg, u_s, u_c = _unpack(params)
    sig_g, sig_r, sig_e, sig_c, sig_s, sig_reg = sigma

    # Likelihood — geographic effect at individual level is α_county
    eta = g0 + u_g[g_idx] + u_r[r_idx] + u_e[e_idx] + u_c[c_idx]
    ll  = np.sum(y * eta - np.logaddexp(0.0, eta))

    # County prior: α_county[c] ~ N(α_state[s[c]] + γ·covariates[c], σ_county²)
    mu_c   = (u_s[county_state_idx]
              + gc * co2_std + gp * pres_std
              + gd * drive_std + gs * samesex_std)
    lp_c   = -0.5 * np.sum((u_c - mu_c) ** 2) / sig_c**2 - n_county * np.log(sig_c)

    # State prior: α_state[s] ~ N(α_region[div[s]], σ_state²)  -- bare nesting
    mu_s   = u_reg[state_div_idx]
    lp_s   = -0.5 * np.sum((u_s - mu_s) ** 2) / sig_s**2 - n_s * np.log(sig_s)

    # Region prior
    lp_reg = -0.5 * np.sum(u_reg ** 2) / sig_reg**2 - n_div * np.log(sig_reg)

    # Demographic priors
    lp_g = -0.5 * np.sum(u_g ** 2) / sig_g**2 - n_g * np.log(sig_g)
    lp_r = -0.5 * np.sum(u_r ** 2) / sig_r**2 - n_r * np.log(sig_r)
    lp_e = -0.5 * np.sum(u_e ** 2) / sig_e**2 - n_e * np.log(sig_e)

    # Hyperpriors
    lp_int   = -0.5 * g0**2 / SIGMA_PRIOR**2
    lp_gamma = -0.5 * (gc**2 + gp**2 + gd**2 + gs**2) / SIGMA_PRIOR**2
    lp_sigma = np.sum(-0.5 * sigma**2 / SIGMA_PRIOR**2 + params[1:7])

    return -(ll + lp_c + lp_s + lp_reg + lp_g + lp_r + lp_e + lp_int + lp_gamma + lp_sigma)

x0 = np.zeros(n_p)
x0[1:7] = np.log(0.5)

start = time.time()
res = minimize(neg_log_posterior, x0, method="L-BFGS-B",
               options={"maxiter": 20_000, "ftol": 1e-9, "gtol": 1e-6})
elapsed = time.time() - start

g0_fit, sigma_fit, gc_fit, gp_fit, gd_fit, gs_fit, \
    u_g_fit, u_r_fit, u_e_fit, u_reg_fit, u_s_fit, u_c_fit = _unpack(res.x)

print(f"Converged : {res.success}  |  iterations: {res.nit}  |  {elapsed:.1f}s")
print(f"Intercept : {g0_fit:.3f}  (baseline prob {expit(g0_fit)*100:.1f}%)")
print(f"γ_carbon  : {gc_fit:.3f}   γ_pres : {gp_fit:.3f}   γ_drive : {gd_fit:.3f}   γ_samesex : {gs_fit:.3f}")
print(f"σ — gender:{sigma_fit[0]:.3f}  race:{sigma_fit[1]:.3f}  educ:{sigma_fit[2]:.3f}  "
      f"county:{sigma_fit[3]:.3f}  state:{sigma_fit[4]:.3f}  region:{sigma_fit[5]:.3f}")
print(f"County-effect range: [{u_c_fit.min():.3f}, {u_c_fit.max():.3f}]")
print(f"State-effect  range: [{u_s_fit.min():.3f}, {u_s_fit.max():.3f}]")

Converged : False  |  iterations: 1  |  0.8s
Intercept : 0.165  (baseline prob 54.1%)
γ_carbon  : -0.000   γ_pres : -0.000   γ_drive : -0.000   γ_samesex : -0.000
σ — gender:0.499  race:0.497  educ:0.497  county:0.001  state:0.451  region:0.492
County-effect range: [-0.003, 0.009]
State-effect  range: [0.000, 0.000]


In [17]:
# ── Predict on county-level poststrat frame ───────────────────────────────
ps = ps_county[["county_fips", "state_fips", "gender", "race4",
                  "educ_category", "N_rounded"]].copy()
ps["educ_category"] = ps["educ_category"].astype(str)

ps_g = pd.Categorical(ps["gender"],        categories=gender_cats).codes
ps_r = pd.Categorical(ps["race4"],         categories=race_cats).codes
ps_e = pd.Categorical(ps["educ_category"], categories=educ_cats).codes
ps_c = pd.Categorical(ps["county_fips"],   categories=county_cats).codes

eta_ps = g0_fit + u_g_fit[ps_g] + u_r_fit[ps_r] + u_e_fit[ps_e] + u_c_fit[ps_c]
ps["predicted_prob"] = expit(eta_ps)

print(f"Predicted probs: min={ps['predicted_prob'].min():.3f}  "
      f"mean={ps['predicted_prob'].mean():.3f}  max={ps['predicted_prob'].max():.3f}")

Predicted probs: min=0.564  mean=0.582  max=0.607


In [18]:
# ── Poststratify by county ────────────────────────────────────────────────
estimates = (
    ps.groupby(["county_fips", "state_fips"])
    .apply(lambda g: np.average(g["predicted_prob"], weights=g["N_rounded"]),
           include_groups=False)
    .reset_index(name="happening_estimate")
)
estimates["state_name"] = estimates["state_fips"].map(STATE_NAMES)
estimates = estimates[["county_fips", "state_fips", "state_name", "happening_estimate"]]

print(f"County estimates: {len(estimates):,}")
print(f"Range: [{estimates['happening_estimate'].min():.4f}, "
      f"{estimates['happening_estimate'].max():.4f}]")
print(f"Mean: {estimates['happening_estimate'].mean():.4f}")

County estimates: 3,143
Range: [0.5776, 0.5981]
Mean: 0.5912


In [19]:
est_dir  = OUTPUT_DIR / "estimates"
diag_dir = OUTPUT_DIR / "diagnostics"
est_dir.mkdir(parents=True, exist_ok=True)
diag_dir.mkdir(parents=True, exist_ok=True)

out_path = est_dir / f"{MODEL_NAME}_county_estimates.csv"
estimates.to_csv(out_path, index=False)
print(f"Saved → {out_path}  (rows: {len(estimates):,})")

diag_path = diag_dir / f"{MODEL_NAME}_county_summary.txt"
with open(diag_path, "w") as f:
    f.write("GLMER-MRP County (Howe 2015, penalised ML) — Diagnostic Summary\n" + "=" * 60 + "\n\n")
    f.write(f"Outcome: {OUTCOME}\n")
    f.write(f"Estimation: L-BFGS-B MAP\n")
    f.write(f"Observations: {n_obs:,}\n")
    f.write(f"Counties: {n_county:,}  States: {n_s}  Divisions: {n_div}\n")
    f.write(f"Converged: {res.success}  iterations: {res.nit}  time: {elapsed:.1f}s\n\n")
    f.write(f"Intercept: {g0_fit:.4f}\n")
    f.write(f"gamma_carbon:  {gc_fit:.4f}\n")
    f.write(f"gamma_pres:    {gp_fit:.4f}\n")
    f.write(f"gamma_drive:   {gd_fit:.4f}\n")
    f.write(f"gamma_samesex: {gs_fit:.4f}\n\n")
    f.write(f"Sigma_gender:  {sigma_fit[0]:.4f}\n")
    f.write(f"Sigma_race:    {sigma_fit[1]:.4f}\n")
    f.write(f"Sigma_educ:    {sigma_fit[2]:.4f}\n")
    f.write(f"Sigma_county:  {sigma_fit[3]:.4f}\n")
    f.write(f"Sigma_state:   {sigma_fit[4]:.4f}\n")
    f.write(f"Sigma_region:  {sigma_fit[5]:.4f}\n\n")
    f.write(f"County estimates — mean:{estimates['happening_estimate'].mean():.4f}  "
            f"min:{estimates['happening_estimate'].min():.4f}  "
            f"max:{estimates['happening_estimate'].max():.4f}\n")
print(f"Diagnostics → {diag_path}")

Saved → ../outputs/estimates/glmer_mrp_county_estimates.csv  (rows: 3,143)
Diagnostics → ../outputs/diagnostics/glmer_mrp_county_summary.txt


In [20]:
# ── State-level rollup diagnostic ──────────────────────────────────────────
# Roll county estimates up to state via population-weighted average, then compare
# with the same model's state-level CSV (from the parallel state notebook).
state_rollup = (
    estimates.merge(
        ps_county.groupby("county_fips")["N_rounded"].sum().reset_index(name="county_pop"),
        on="county_fips", how="left",
    )
    .dropna(subset=["happening_estimate"])
    .groupby("state_fips")
    .apply(lambda g: np.average(g["happening_estimate"], weights=g["county_pop"]),
           include_groups=False)
    .reset_index(name="county_rollup")
)
state_rollup["state_name"] = state_rollup["state_fips"].map(STATE_NAMES)

state_csv = OUTPUT_DIR / "estimates" / f"{STATE_CSV_NAME}"
if state_csv.exists():
    state_est = pd.read_csv(state_csv, dtype={"state_fips": str})
    cmp = state_rollup.merge(state_est[["state_fips", "estimate"]], on="state_fips", how="left")
    cmp["abs_diff"] = (cmp["county_rollup"] - cmp["estimate"]).abs()
    print(f"\nState rollup vs {STATE_CSV_NAME}:")
    print(f"  mean |diff|: {cmp['abs_diff'].mean():.4f}")
    print(f"  max  |diff|: {cmp['abs_diff'].max():.4f}")
    print(cmp[["state_fips", "state_name", "county_rollup", "estimate", "abs_diff"]]
          .sort_values("abs_diff", ascending=False).head(10).to_string(index=False))
else:
    print(f"\nNo state CSV found at {state_csv} — skipping rollup comparison.")


State rollup vs glmer_mrp_state_estimates.csv:
  mean |diff|: 0.0722
  max  |diff|: 0.2352
state_fips           state_name  county_rollup  estimate  abs_diff
        15               Hawaii       0.583379  0.818565  0.235186
        11 District of Columbia       0.589895  0.805463  0.215568
        24             Maryland       0.589372  0.792744  0.203372
        53           Washington       0.592231  0.763937  0.171705
        02               Alaska       0.589426  0.751279  0.161853
        06           California       0.586051  0.744357  0.158306
        41               Oregon       0.592293  0.737674  0.145380
        51             Virginia       0.590571  0.722283  0.131712
        27            Minnesota       0.593638  0.704642  0.111005
        50              Vermont       0.595778  0.706156  0.110378
